## 1) Load both files

In [2]:
import pandas as pd
import numpy as np
import os

pm25 = pd.read_csv("../../data/validated/PM25_cityday.csv")
o3   = pd.read_csv("../../data/validated/O3_cityday.csv")

print("PM25:", pm25.shape)
print("O3:", o3.shape)

pm25.head()

PM25: (256216, 5)
O3: (267124, 5)


,City,Date,PM25_daily,Year,Month
0,MontrÃ©al,2020-01-01,4.333333,2020,1
1,MontrÃ©al,2020-01-02,7.625000,2020,1
2,MontrÃ©al,2020-01-03,10.750000,2020,1
3,MontrÃ©al,2020-01-04,4.416667,2020,1
4,MontrÃ©al,2020-01-05,3.833333,2020,1


## 2) Fix Date type + clean City text

In [5]:
# convert Date to datetime
pm25["Date"] = pd.to_datetime(pm25["Date"], errors="coerce")
o3["Date"]   = pd.to_datetime(o3["Date"], errors="coerce")

# clean City (remove extra spaces)
pm25["City"] = pm25["City"].astype(str).str.strip()
o3["City"]   = o3["City"].astype(str).str.strip()

print("Invalid PM25 dates:", pm25["Date"].isna().sum())
print("Invalid O3 dates:", o3["Date"].isna().sum())

Invalid PM25 dates: 0
Invalid O3 dates: 0


## 3) Convert PM25_daily and O3_daily to numeric

In [8]:
pm25["PM25_daily"] = pd.to_numeric(pm25["PM25_daily"], errors="coerce")
o3["O3_daily"]     = pd.to_numeric(o3["O3_daily"], errors="coerce")

print("PM25_daily NaNs:", pm25["PM25_daily"].isna().sum())
print("O3_daily NaNs:", o3["O3_daily"].isna().sum())

pm25 = pm25.dropna(subset=["PM25_daily"])
o3   = o3.dropna(subset=["O3_daily"])

PM25_daily NaNs: 0
O3_daily NaNs: 0


## 4) Drop duplicates correctly

In [11]:
print("PM25 duplicates:", pm25.duplicated(subset=["City","Date"]).sum())
print("O3 duplicates:", o3.duplicated(subset=["City","Date"]).sum())

pm25 = pm25.drop_duplicates(subset=["City","Date"])
o3   = o3.drop_duplicates(subset=["City","Date"])

print("After drop -> PM25 duplicates:", pm25.duplicated(subset=["City","Date"]).sum())
print("After drop -> O3 duplicates:", o3.duplicated(subset=["City","Date"]).sum())

PM25 duplicates: 0
O3 duplicates: 0
After drop -> PM25 duplicates: 0
After drop -> O3 duplicates: 0


## 5) Merge

In [14]:
df = pm25.merge(o3, on=["City","Date"], how="inner")

print("Merged shape:", df.shape)
df.head()

Merged shape: (230625, 8)


,City,Date,PM25_daily,Year_x,Month_x,O3_daily,Year_y,Month_y
0,MontrÃ©al,2020-01-01,4.333333,2020,1,18.458333,2020,1
1,MontrÃ©al,2020-01-02,7.625000,2020,1,24.125000,2020,1
2,MontrÃ©al,2020-01-03,10.750000,2020,1,11.791667,2020,1
3,MontrÃ©al,2020-01-04,4.416667,2020,1,23.666667,2020,1
4,MontrÃ©al,2020-01-05,3.833333,2020,1,29.166667,2020,1


## 6) Create Year + Month + Season from Date

In [17]:
df["Year"]  = df["Date"].dt.year
df["Month"] = df["Date"].dt.month

def season_from_month(m):
    if m in [12,1,2]:  return "Winter"
    if m in [3,4,5]:   return "Spring"
    if m in [6,7,8]:   return "Summer"
    return "Fall"

df["Season"] = df["Month"].apply(season_from_month)

df[["City","Date","PM25_daily","O3_daily","Year","Month","Season"]].head()

,City,Date,PM25_daily,O3_daily,Year,Month,Season
0,MontrÃ©al,2020-01-01,4.333333,18.458333,2020,1,Winter
1,MontrÃ©al,2020-01-02,7.625000,24.125000,2020,1,Winter
2,MontrÃ©al,2020-01-03,10.750000,11.791667,2020,1,Winter
3,MontrÃ©al,2020-01-04,4.416667,23.666667,2020,1,Winter
4,MontrÃ©al,2020-01-05,3.833333,29.166667,2020,1,Winter


## 7) Validation checks

In [20]:
print("Unique cities:", df["City"].nunique())
print("Year range:", df["Year"].min(), "-", df["Year"].max())

print(df[["PM25_daily","O3_daily"]].describe())

Unique cities: 169
Year range: 2020 - 2023
          PM25_daily       O3_daily
count  230625.000000  230625.000000
mean        7.017622      26.042728
std        11.567285       9.403205
min         0.000000       0.000000
25%         3.333333      19.458333
50%         5.000000      26.083333
75%         7.666667      32.708333
max       595.625000      69.000000


## 8) Save output

In [23]:
os.makedirs("../../data/processed", exist_ok=True)
df.to_csv("../../data/processed/pm25_o3_merged.csv", index=False)
print("Saved: data/processed/pm25_o3_merged.csv")

Saved: data/processed/pm25_o3_merged.csv


In [25]:
df.sort_values("PM25_daily", ascending=False).head(10)


,City,Date,PM25_daily,Year_x,Month_x,O3_daily,Year_y,Month_y,Year,Month,Season
186407,Kamloops,2023-08-21,595.625000,2023,8,17.086957,2023,8,2023,8,Summer
209825,Senneterre,2023-06-25,593.333333,2023,6,50.136364,2023,6,2023,6,Summer
209824,Senneterre,2023-06-24,514.416667,2023,6,54.227273,2023,6,2023,6,Summer
213196,Buffalo Narrows,2023-05-22,504.166667,2023,5,26.750000,2023,5,2023,5,Spring
191739,Fort St. John,2023-05-19,461.000000,2023,5,22.913043,2023,5,2023,5,Spring
207013,La Dore,2023-06-25,458.250000,2023,6,31.875000,2023,6,2023,6,Summer
203433,Saguenay,2023-06-25,456.791667,2023,6,37.666667,2023,6,2023,6,Summer
209806,Senneterre,2023-06-03,436.333333,2023,6,31.291667,2023,6,2023,6,Summer
30399,Castlegar,2020-09-13,423.541667,2020,9,15.391304,2020,9,2020,9,Fall
180551,Drayton Valley,2023-05-20,408.500000,2023,5,38.565217,2023,5,2023,5,Spring
